In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql
use catalog lendingclub;
create schema if not exists silver_enriched;
use schema silver_enriched;
select current_catalog(), current_schema();

In [0]:
display(spark.sql('describe extended silver_cleaned.loans_defaulter_public_record'))

In [0]:
#Ideally we should have 1 member_id in table. validate if we have more 
#customer_data

spark.sql('''
        select member_id, count(*) as total_count
        from silver_cleaned.loans_defaulter_public_record
        group by member_id
        order by total_count desc''').show()

In [0]:
bad_cust_publicRec_df = spark.sql('''
                        select member_id
                        from (select member_id, count(*) as total_count
                        from silver_cleaned.loans_defaulter_public_record
                        group by member_id
                        having total_count > 1)''')

In [0]:
bad_cust_publicRec_df.count()

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/bad_data/LoanDefaulterPublicRecord/", recurse=True)

In [0]:
bad_cust_publicRec_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/bad_data/LoanDefaulterPublicRecord/')

##Seggregate bad data 

In [0]:
bad_cust_publicRec_df.createOrReplaceTempView('bad_data')

In [0]:
#seggregate bad data from actual cleaned dataset

loan_defaulter_publicRecord_df = spark.sql('''
                        select * from silver_cleaned.loans_defaulter_public_record 
                        where member_id not in 
                        (select member_id from bad_data)''')

In [0]:
dbutils.fs.rm("/Volumes/lendingclub/storagelocation/final_cleaned/loans_defaulter_publicRecord/", recurse=True)

In [0]:
loan_defaulter_publicRecord_df.write.format('delta').mode('overwrite').save('/Volumes/lendingclub/storagelocation/final_cleaned/loans_defaulter_publicRecord/')

In [0]:
%sql
create or replace table silver_enriched.loans_defaulter_publicRecord
as
select * 
from delta.`/Volumes/lendingclub/storagelocation/final_cleaned/loans_defaulter_publicRecord/`

In [0]:
#check  if there is any member_id is repeating

spark.sql('''select member_id, count(*) as total
          from silver_enriched.loans_defaulter_publicRecord
          group by member_id 
          order by total desc''').show()